<a href="https://colab.research.google.com/github/MilesEhrlich/Adversarial-Robustness-Research/blob/main/Baseline_Human_4_23_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MNIST Multitask: Baseline vs. Human-in-the-Loop

This notebook implements and compares two training conditions from the proposal:

| Condition | Training data |
|-----------|---------------|
| **Part 1 — Baseline** | Basic Augmentation, no quality gate |
| **Part 2 — Human** | Only human-approved augmented candidates |

**How to run**
- **Baseline only** → run *Shared Setup* cells, then *Part 1* cells.
- **Both conditions** → run *Shared Setup* → *Part 1* → *Part 2* → *Part 3*.
- The review widget in Part 2 Step 2 is interactive — click the buttons to approve or reject each candidate.
- All outputs (models, plots) are saved in the working directory.

---
## Shared Setup
Run all cells in this section before running either Part 1 or Part 2.

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import os, json, sys

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow {tf.__version__}  |  NumPy {np.__version__}')

In [ ]:
# ── architecture ───────────────────────────────────────────────────
LATENT_DIM = 128
BETA       = 0.3        # reconstruction loss weight (eq. 2 in proposal)

# ── training ───────────────────────────────────────────────────────
BATCH_SIZE      = 64
M_VIEWS         = 2     # augmented copies per original (baseline generator)
EPOCHS_BASELINE = 10    # epochs for Part 1
EPOCHS_HUMAN    = 10    # epochs for Part 2  (match baseline for fair comparison)
STEPS_PER_EPOCH = 200   # ~200 steps x 64 batch = ~13k images/epoch (full MNIST = ~937 steps)
VAL_STEPS       = 50
LR              = 1e-3

# ── candidate generation (Part 2 Step 1) ──────────────────────────
N_ORIGINALS = 500        # source images to draw augmented candidates from

# ── paths ──────────────────────────────────────────────────────────
OUT_DIR       = os.getcwd()
CAND_DIR      = os.path.join(OUT_DIR, 'candidates')
CAND_PATH     = os.path.join(CAND_DIR, 'candidates.npz')
STATE_PATH    = os.path.join(CAND_DIR, 'review_state.json')
APPROVED_PATH = os.path.join(CAND_DIR, 'approved.npz')

# accumulates results for the comparison in Part 3
results = {}

print(f'Output directory: {OUT_DIR}')

In [ ]:
# ── data loading ───────────────────────────────────────────────────

def load_mnist():
    print('loading mnist...')
    (x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
    x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
    x_test  = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0
    y_train = y_train.astype('int32')
    y_test  = y_test.astype('int32')
    print(f'train: {x_train.shape}   test: {x_test.shape}')
    return x_train, x_test, y_train, y_test


# ── vectorized preprocessing (operates on whole batches at once) ───
# These replace the per-image Python loops that made the old generator slow.

def segment_batch(imgs):
    """Threshold-based background removal for a batch (N, 28, 28, 1)."""
    mask = (imgs[:, :, :, 0] > 0.1).astype('float32')          # (N, 28, 28)
    return (imgs[:, :, :, 0] * mask)[..., np.newaxis].astype('float32')


def augment_batch(imgs, rng):
    """Apply random augmentations to a batch (N, 28, 28, 1) — all ops vectorized."""
    n   = len(imgs)
    out = imgs.copy()

    # random horizontal flip
    flip = rng.random(n) < 0.5
    out[flip] = out[flip, :, ::-1, :]

    # random crop: pad to 32x32, then crop a random 28x28 window per image
    pad    = 2
    padded = np.pad(out, ((0, 0), (pad, pad), (pad, pad), (0, 0)), mode='constant')
    h_offs = rng.integers(0, 2 * pad + 1, size=n)
    w_offs = rng.integers(0, 2 * pad + 1, size=n)
    for i in range(n):                          # loop only over crop offsets, not pixel ops
        out[i] = padded[i, h_offs[i]:h_offs[i]+28, w_offs[i]:w_offs[i]+28, :]

    # brightness jitter — vectorized multiply across the whole batch
    factors = 1.0 + rng.uniform(-0.10, 0.10, size=(n, 1, 1, 1))
    out = np.clip(out * factors, 0.0, 1.0)

    # random erasing — only ~30% of images, so the loop is short
    for i in np.where(rng.random(n) < 0.3)[0]:
        eh = rng.integers(4, 10);  ew = rng.integers(4, 10)
        ey = rng.integers(0, 28 - eh);  ex = rng.integers(0, 28 - ew)
        out[i, ey:ey+eh, ex:ex+ew, :] = 0.0

    return out.astype('float32')

In [ ]:
def build_model(latent_dim=LATENT_DIM, num_classes=10):
    img_in = Input(shape=(28, 28, 1), name='image_input')

    # shared encoder (lines 2-3 of Algorithm 1)
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(img_in)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)   # 28x28 -> 14x14

    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)   # 14x14 -> 7x7

    x = layers.Flatten()(x)
    x = layers.Dense(latent_dim, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)     # latent vector

    # classification head  p(x) in proposal
    dx = layers.Dense(64, use_bias=False)(x)
    dx = layers.BatchNormalization()(dx)
    dx = layers.Activation('relu')(dx)
    dx = layers.Dropout(0.3)(dx)
    class_out = layers.Dense(num_classes, activation='softmax', name='class_output')(dx)

    # decoder / identity head  p'(x) in proposal
    gx = layers.Dense(7 * 7 * 64, use_bias=False)(x)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)
    gx = layers.Reshape((7, 7, 64))(gx)
    gx = layers.Conv2DTranspose(64, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)   # 7x7 -> 14x14
    gx = layers.Conv2DTranspose(32, (3, 3), strides=(2, 2), padding='same', use_bias=False)(gx)
    gx = layers.BatchNormalization()(gx)
    gx = layers.Activation('relu')(gx)   # 14x14 -> 28x28
    recon_out = layers.Conv2D(1, (3, 3), padding='same', activation='sigmoid',
                              name='decoder_output')(gx)

    model = Model(inputs=img_in, outputs=[class_out, recon_out], name='MultitaskBaseline')
    model.compile(
        optimizer=Adam(LR),
        loss={
            'class_output'  : 'sparse_categorical_crossentropy',
            'decoder_output': 'binary_crossentropy',
        },
        loss_weights={
            'class_output'  : 1.0,
            'decoder_output': BETA / 784.0,  # normalise by pixel count so BETA=1 = equal weight
        },
        metrics={
            'class_output'  : ['accuracy'],
            'decoder_output': ['mse'],
        },
    )
    return model

In [ ]:
def _wrap(gen):
    """Add decoder target (= input image) to each batch."""
    for imgs, labels in gen:
        yield imgs, {'class_output': labels, 'decoder_output': imgs}


def train_model(model, train_gen, val_gen, epochs, steps_per_epoch=STEPS_PER_EPOCH, val_steps=VAL_STEPS):
    return model.fit(
        _wrap(train_gen),
        steps_per_epoch  = steps_per_epoch,
        validation_data  = _wrap(val_gen),
        validation_steps = val_steps,
        epochs           = epochs,
        verbose          = 1,
    )


def evaluate(model, x_test, y_test):
    print('\nevaluating on test set...')
    imgs, labels = x_test[:2000], y_test[:2000]
    class_probs, recons = model.predict(imgs, batch_size=128, verbose=0)
    preds     = np.argmax(class_probs, axis=1)
    accuracy  = float(np.mean(preds == labels))
    recon_mse = float(np.mean((imgs - recons) ** 2))
    print(f'  accuracy  : {accuracy * 100:.2f}%')
    print(f'  recon mse : {recon_mse:.6f}')
    return accuracy, recon_mse


def plot_reconstructions(model, x_test, y_test, n=8, save_prefix=''):
    idx      = np.random.choice(len(x_test), n, replace=False)
    sample_x = x_test[idx]
    sample_y = y_test[idx]
    class_probs, recons = model.predict(sample_x, verbose=0)
    preds = np.argmax(class_probs, axis=1)

    fig, axes = plt.subplots(2, n, figsize=(2.5 * n, 5))
    fig.suptitle('original (top) vs reconstruction (bottom) -- green=correct  red=wrong', fontsize=11)
    for col in range(n):
        orig, recon = sample_x[col], recons[col]
        pred, true  = preds[col], sample_y[col]
        mse = float(np.mean((orig - recon) ** 2))
        axes[0, col].imshow(orig[:, :, 0],  cmap='gray', vmin=0, vmax=1)
        axes[0, col].set_title(str(true), fontsize=8)
        axes[0, col].axis('off')
        axes[1, col].imshow(recon[:, :, 0], cmap='gray', vmin=0, vmax=1)
        color = 'green' if pred == true else 'red'
        axes[1, col].set_title(f'{pred}  mse={mse:.4f}', fontsize=7, color=color)
        axes[1, col].axis('off')
    axes[0, 0].set_ylabel('original',      fontsize=9)
    axes[1, 0].set_ylabel('reconstructed', fontsize=9)
    plt.tight_layout()
    if save_prefix:
        path = os.path.join(OUT_DIR, f'{save_prefix}reconstructions.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'saved -> {path}')
    plt.show()


def plot_history(history, save_prefix=''):
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle('training history', fontsize=12)

    axes[0].plot(history.history['loss'],     label='train')
    axes[0].plot(history.history['val_loss'], label='val')
    axes[0].set_title('total loss'); axes[0].set_xlabel('epoch'); axes[0].legend()

    axes[1].plot(history.history['class_output_accuracy'],     label='train')
    axes[1].plot(history.history['val_class_output_accuracy'], label='val')
    axes[1].set_title('classification accuracy')
    axes[1].set_xlabel('epoch'); axes[1].set_ylim(0, 1); axes[1].legend()

    axes[2].plot(history.history['decoder_output_mse'],     label='train')
    axes[2].plot(history.history['val_decoder_output_mse'], label='val')
    axes[2].set_title('reconstruction mse'); axes[2].set_xlabel('epoch'); axes[2].legend()

    plt.tight_layout()
    if save_prefix:
        path = os.path.join(OUT_DIR, f'{save_prefix}training_curves.png')
        plt.savefig(path, dpi=150, bbox_inches='tight')
        print(f'saved -> {path}')
    plt.show()

In [ ]:
x_train, x_test, y_train, y_test = load_mnist()

In [ ]:
def make_batches(x, y, batch_size=BATCH_SIZE, m=M_VIEWS):
    """Infinite generator using vectorized segment + augment (lines 6-10 of Algorithm 1)."""
    rng = np.random.default_rng(SEED)
    n   = len(x)
    while True:
        idx        = rng.integers(0, n, size=batch_size)
        orig       = segment_batch(x[idx])
        aug_views  = [augment_batch(orig, rng) for _ in range(m)]
        imgs       = np.concatenate([orig] + aug_views, axis=0)
        labels     = np.tile(y[idx], m + 1)
        perm       = rng.permutation(len(imgs))
        yield imgs[perm], labels[perm]

---
## Part 1 — Baseline Model

Augmentation with no quality gate.  
Implements Algorithm 1 from the proposal exactly as written.

**To run baseline only**: execute the Shared Setup cells above, then run every cell in this section.

In [ ]:
# make_batches is defined in Shared Setup above — nothing to run here.

In [ ]:
print('=' * 60)
print('PART 1 — BASELINE MODEL  (on-the-fly augmentation)')
print('=' * 60)

baseline_model = build_model()
baseline_model.summary()

baseline_history = train_model(
    baseline_model,
    make_batches(x_train, y_train),
    make_batches(x_test,  y_test),
    epochs=EPOCHS_BASELINE,
)

In [ ]:
baseline_acc, baseline_mse = evaluate(baseline_model, x_test, y_test)
results['baseline'] = {'accuracy': baseline_acc, 'recon_mse': baseline_mse}

plot_reconstructions(baseline_model, x_test, y_test, n=8, save_prefix='baseline_')
plot_history(baseline_history, save_prefix='baseline_')

In [ ]:
  from sklearn.metrics import classification_report, confusion_matrix

  _imgs, _labels = x_test[:2000], y_test[:2000]
  _probs, _ = baseline_model.predict(_imgs, batch_size=128, verbose=0)
  _preds = np.argmax(_probs, axis=1)

  print('Baseline — per-digit accuracy')
  print('-' * 36)
  for digit in range(10):
      mask = _labels == digit
      acc  = np.mean(_preds[mask] == _labels[mask]) * 100
      print(f'  Digit {digit} : {acc:6.2f}%  ({mask.sum()} samples)')

  # full classification report
  print('\nBaseline — classification report')
  print(classification_report(_labels, _preds, target_names=[str(i) for i in range(10)]))



---
## Part 2 — Human Section

Augmented candidates are manually approved before they enter the training set.

1. **Generate candidates** — produce a pool of augmented images
2. **Review candidates** — approve or reject each one using the widget below
3. **Train on approved set** — only approved samples reach the model

> Originals (segmented MNIST images) bypass the gate and are always included as anchors,  
> exactly as described in the proposal framework.

### Step 1 — Generate Candidates

Draws `N_ORIGINALS` images from the MNIST training set, applies `M_VIEWS` augmentations to each,  
and saves the pool to `candidates/candidates.npz`.  
Re-running this cell resets review progress.

In [ ]:
def generate_candidates():
      os.makedirs(CAND_DIR, exist_ok=True)

      rng            = np.random.default_rng(SEED)
      source_indices = rng.choice(len(x_train), size=N_ORIGINALS, replace=False)

      src_imgs   = segment_batch(x_train[source_indices])
      src_labels = y_train[source_indices]

      cand_imgs, cand_labels, cand_source_idx, cand_orig_imgs = [], [], [], []

      total = N_ORIGINALS * M_VIEWS
      print(f'generating {N_ORIGINALS} originals x {M_VIEWS} views = {total} candidates...')

      for j, src_idx in enumerate(source_indices):
          orig  = src_imgs[j]
          label = src_labels[j]
          for _ in range(M_VIEWS):
              aug = augment_batch(orig[np.newaxis], rng)[0]
              cand_imgs.append(aug)
              cand_labels.append(label)
              cand_source_idx.append(int(src_idx))
              cand_orig_imgs.append(orig)

      cand_imgs       = np.stack(cand_imgs,       axis=0)
      cand_labels     = np.array(cand_labels,     dtype='int32')
      cand_source_idx = np.array(cand_source_idx, dtype='int32')
      cand_orig_imgs  = np.stack(cand_orig_imgs,  axis=0)

      np.savez_compressed(CAND_PATH,
          imgs=cand_imgs, labels=cand_labels,
          source_idx=cand_source_idx, orig_imgs=cand_orig_imgs)
      print(f'saved {len(cand_imgs)} candidates -> {CAND_PATH}')
      print(f'  label distribution: {np.bincount(cand_labels)}')

      n_show   = min(80, len(cand_imgs))
      show_idx = rng.choice(len(cand_imgs), size=n_show, replace=False)
      rows, cols = 8, 10
      fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.4, rows * 1.4), facecolor='#1a1a2e')
      fig.suptitle(f'Candidate contact sheet  ({n_show} of {len(cand_imgs)} shown)',
                   fontsize=10, color='white')
      for i, ax in enumerate(axes.flat):
          ax.set_facecolor('#16213e')
          if i < n_show:
              ci = show_idx[i]
              ax.imshow(cand_imgs[ci, :, :, 0], cmap='gray', vmin=0, vmax=1)
              ax.set_title(str(cand_labels[ci]), fontsize=7, color='#a0d0ff', pad=2)
          ax.axis('off')
      plt.tight_layout()
      cs_path = os.path.join(CAND_DIR, 'contact_sheet.png')
      plt.savefig(cs_path, dpi=130, bbox_inches='tight', facecolor='#1a1a2e')
      plt.show()
      print(f'contact sheet saved -> {cs_path}')
      print(f'\nNext: run the review widget below.')

generate_candidates()

### Step 2 — Review Candidates

Each candidate is shown alongside its source image.  
Use the buttons to approve or reject. Progress is saved after every decision  
so you can close and resume at any point.

| Button | Action |
|--------|--------|
| ✓ Approve | Accept this augmentation into the training set |
| ✗ Reject  | Discard this augmentation |
| ↩ Undo    | Step back one decision |
| 💾 Save & Done | Write `approved.npz` and stop |

Reviewing all candidates automatically writes `approved.npz`.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
except ImportError:
    raise ImportError("ipywidgets is required for the review UI.  Run:  pip install ipywidgets")

if not os.path.exists(CAND_PATH):
    print('ERROR: candidates.npz not found. Run the Generate Candidates cell first.')
else:
    _raw  = np.load(CAND_PATH)
    _data = {k: _raw[k] for k in _raw.files}
    _n    = len(_data['imgs'])

    # load or initialise review state
    if os.path.exists(STATE_PATH):
        with open(STATE_PATH) as f:
            _saved = json.load(f)
        _rev = {'idx': _saved['current_idx'], 'decisions': list(_saved['decisions'])}
        print(f'Resuming — {_rev["idx"]} of {_n} decisions already recorded.')
    else:
        _rev = {'idx': 0, 'decisions': []}
        print(f'Starting fresh review of {_n} candidates.')

    # ── widget helpers ─────────────────────────────────────────────

    def _rev_save():
        state = {'current_idx': _rev['idx'], 'decisions': _rev['decisions']}
        with open(STATE_PATH, 'w') as f:
            json.dump(state, f, indent=2)

    def _rev_finalize():
        decisions  = _rev['decisions']
        n_dec      = len(decisions)
        n_app      = sum(1 for d in decisions if d == 1)
        approved_i = [i for i, d in enumerate(decisions) if d == 1]
        if approved_i:
            np.savez_compressed(APPROVED_PATH,
                                imgs=_data['imgs'][approved_i],
                                labels=_data['labels'][approved_i])
        pct = 100 * n_app / n_dec if n_dec else 0
        lbl_status.value = (
            f'<b>Done!</b>  Reviewed: {n_dec}  '
            f'| <span style="color:green">Approved: {n_app} ({pct:.0f}%)</span>  '
            f'| <span style="color:red">Rejected: {n_dec - n_app}</span>  '
            f'<br><b>approved.npz saved.</b>  Ready to train — run Step 3.')

    # ── widgets ────────────────────────────────────────────────────

    _W = widgets.Layout
    out_img    = widgets.Output()
    lbl_prog   = widgets.HTML()
    lbl_status = widgets.HTML()
    btn_a      = widgets.Button(description='✓ Approve',    button_style='success', layout=_W(width='130px'))
    btn_r      = widgets.Button(description='✗ Reject',     button_style='danger',  layout=_W(width='130px'))
    btn_u      = widgets.Button(description='↩ Undo',       button_style='warning', layout=_W(width='130px'))
    btn_done   = widgets.Button(description='💾 Save & Done', button_style='info',  layout=_W(width='140px'))

    def _rev_update_ui():
        idx       = _rev['idx']
        decisions = _rev['decisions']
        n_dec     = len(decisions)
        n_app     = sum(1 for d in decisions if d == 1)

        lbl_prog.value = (
            f'<b>Candidate {idx + 1} / {_n}</b> &nbsp;&nbsp; '
            f'Approved: <b style="color:green">{n_app}</b> &nbsp; '
            f'Rejected: <b style="color:red">{n_dec - n_app}</b> &nbsp; '
            f'Remaining: <b>{_n - idx}</b>')

        with out_img:
            clear_output(wait=True)
            if idx >= _n:
                print('All candidates reviewed!')
                return
            img_orig = _data['orig_imgs'][idx, :, :, 0]
            img_aug  = _data['imgs'][idx,      :, :, 0]
            label    = int(_data['labels'][idx])
            fig, axes = plt.subplots(1, 2, figsize=(5, 2.8))
            axes[0].imshow(img_orig, cmap='gray', vmin=0, vmax=1)
            axes[0].set_title(f'Original  (digit {label})', fontsize=9)
            axes[0].axis('off')
            axes[1].imshow(img_aug,  cmap='gray', vmin=0, vmax=1)
            axes[1].set_title('Augmented candidate', fontsize=9)
            axes[1].axis('off')
            plt.tight_layout()
            plt.show()

    def _on_approve(b):
        if _rev['idx'] >= _n: return
        _rev['decisions'].append(1)
        _rev['idx'] += 1
        _rev_save()
        lbl_status.value = '<span style="color:green">✓ Approved</span>'
        if _rev['idx'] >= _n:
            _rev_finalize()
        else:
            _rev_update_ui()

    def _on_reject(b):
        if _rev['idx'] >= _n: return
        _rev['decisions'].append(-1)
        _rev['idx'] += 1
        _rev_save()
        lbl_status.value = '<span style="color:red">✗ Rejected</span>'
        if _rev['idx'] >= _n:
            _rev_finalize()
        else:
            _rev_update_ui()

    def _on_undo(b):
        if not _rev['decisions']: return
        _rev['decisions'].pop()
        _rev['idx'] = max(0, _rev['idx'] - 1)
        _rev_save()
        lbl_status.value = '↩ Undone'
        _rev_update_ui()

    def _on_done(b):
        _rev_save()
        _rev_finalize()

    btn_a.on_click(_on_approve)
    btn_r.on_click(_on_reject)
    btn_u.on_click(_on_undo)
    btn_done.on_click(_on_done)

    ui = widgets.VBox([
        lbl_prog,
        out_img,
        widgets.HBox([btn_a, btn_r, btn_u, btn_done]),
        lbl_status,
    ])
    display(ui)
    _rev_update_ui()

### After Review — Inspect Approved Set

Run this cell after finishing the review widget above to see a summary and sample of approved images.

In [ ]:
if not os.path.exists(APPROVED_PATH):
    print('approved.npz not found — complete the review widget above first.')
else:
    _appr     = np.load(APPROVED_PATH)
    appr_imgs = _appr['imgs']
    appr_labs = _appr['labels']
    n_approved = len(appr_imgs)
    total_cand = N_ORIGINALS * M_VIEWS

    print(f'Approved set summary')
    print(f'  total candidates : {total_cand}')
    print(f'  approved         : {n_approved}  ({100 * n_approved / total_cand:.1f}%)')
    print(f'  rejected         : {total_cand - n_approved}')
    print(f'  label distribution: {np.bincount(appr_labs)}')

    # show a grid of approved samples
    n_show   = min(32, n_approved)
    show_idx = np.random.choice(n_approved, n_show, replace=False)
    cols = 8
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.4, rows * 1.4))
    fig.suptitle(f'Sample approved augmentations  ({n_show} of {n_approved})', fontsize=10)
    for i, ax in enumerate(axes.flat):
        if i < n_show:
            ax.imshow(appr_imgs[show_idx[i], :, :, 0], cmap='gray', vmin=0, vmax=1)
            ax.set_title(str(appr_labs[show_idx[i]]), fontsize=7)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

### Step 3 — Train on Approved Samples

The batch generator serves a 50/50 mix of:
- **Original segmented images** — always included, bypass the gate
- **Human-approved augmented images** — sampled from `approved.npz`

In [ ]:
def make_approved_batches(x_train, y_train, approved_path, batch_size=BATCH_SIZE):
    """50/50 mix of segmented originals and human-approved augmented samples."""
    raw        = np.load(approved_path)
    aug_imgs   = raw['imgs']    # (N_approved, 28, 28, 1)
    aug_labels = raw['labels']  # (N_approved,)
    n_aug      = len(aug_imgs)
    n_train    = len(x_train)
    rng        = np.random.default_rng(SEED)
    half       = max(1, batch_size // 2)

    print(f'  [human-supervised] {n_aug} approved augmented samples')
    print(f'  [human-supervised] {n_train} original MNIST images as anchors')

    while True:
        orig_idx  = rng.integers(0, n_train, size=half)
        orig_imgs = segment_batch(x_train[orig_idx])   # vectorized
        orig_labs = y_train[orig_idx]

        aug_idx   = rng.integers(0, n_aug, size=half)
        batch_aug = aug_imgs[aug_idx]
        aug_labs  = aug_labels[aug_idx]

        imgs   = np.concatenate([orig_imgs, batch_aug], axis=0)
        labels = np.concatenate([orig_labs, aug_labs],  axis=0)
        perm   = rng.permutation(len(imgs))
        yield imgs[perm], labels[perm]

In [ ]:
if not os.path.exists(APPROVED_PATH):
    print('ERROR: approved.npz not found. Complete the review widget (Step 2) first.')
else:
    print('=' * 60)
    print('PART 2 — HUMAN-IN-THE-LOOP MODEL  (approved augmentations only)')
    print('=' * 60)

    human_model = build_model()
    human_model.summary()

    human_history = train_model(
        human_model,
        make_approved_batches(x_train, y_train, APPROVED_PATH),
        make_batches(x_test, y_test),
        epochs=EPOCHS_HUMAN,
    )

In [ ]:
if 'human_model' not in dir():
    print('Train the human model first (cell above).')
else:
    human_acc, human_mse = evaluate(human_model, x_test, y_test)
    results['human'] = {'accuracy': human_acc, 'recon_mse': human_mse}

    plot_reconstructions(human_model, x_test, y_test, n=8, save_prefix='human_')
    plot_history(human_history, save_prefix='human_')

In [ ]:
  from sklearn.metrics import classification_report, confusion_matrix

  _imgs, _labels = x_test[:2000], y_test[:2000]
  _probs, _ = human_model.predict(_imgs, batch_size=128, verbose=0)
  _preds = np.argmax(_probs, axis=1)

  # per-digit accuracy table
  print('-' * 36)
  for digit in range(10):
      mask = _labels == digit
      acc  = np.mean(_preds[mask] == _labels[mask]) * 100
      print(f'  Digit {digit} : {acc:6.2f}%  ({mask.sum()} samples)')

  # full classification report
  print('\nHuman — classification report')
  print(classification_report(_labels, _preds, target_names=[str(i) for i in range(10)]))

---
## Part 3 — Comparison

Run this section after completing both Part 1 and Part 2.  
Compares test-set accuracy and reconstruction MSE side-by-side.

In [ ]:
if len(results) < 2:
    missing = [k for k in ('baseline', 'human') if k not in results]
    print(f'Missing results for: {missing}. Run the corresponding Part first.')
else:
    b = results['baseline']
    h = results['human']

    # ── text summary ───────────────────────────────────────────────
    print('=' * 55)
    print(f'{"":30s}  {"Baseline":>10s}  {"Human":>10s}')
    print('-' * 55)
    print(f'{"Test accuracy":30s}  {b["accuracy"]*100:>9.2f}%  {h["accuracy"]*100:>9.2f}%')
    print(f'{"Reconstruction MSE":30s}  {b["recon_mse"]:>10.6f}  {h["recon_mse"]:>10.6f}')
    delta_acc = (h['accuracy'] - b['accuracy']) * 100
    delta_mse = h['recon_mse'] - b['recon_mse']
    direction_acc = 'improvement' if delta_acc > 0 else 'decline'
    direction_mse = 'improvement' if delta_mse < 0 else 'increase'
    print('-' * 55)
    print(f'  Accuracy delta : {delta_acc:+.2f}%  ({direction_acc})')
    print(f'  MSE delta      : {delta_mse:+.6f}  ({direction_mse})')
    print('=' * 55)

    # ── bar chart ──────────────────────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle('Baseline vs. Human-in-the-Loop', fontsize=13, fontweight='bold')

    labels = ['Baseline', 'Human']
    colors = ['#4c72b0', '#dd8452']

    accs = [b['accuracy'] * 100, h['accuracy'] * 100]
    bars1 = ax1.bar(labels, accs, color=colors, width=0.5)
    ax1.set_ylabel('Test Accuracy (%)')
    ax1.set_title('Classification Accuracy')
    ax1.set_ylim(max(0, min(accs) - 5), min(100, max(accs) + 5))
    for bar, v in zip(bars1, accs):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                 f'{v:.2f}%', ha='center', va='bottom', fontsize=11)

    mses = [b['recon_mse'], h['recon_mse']]
    bars2 = ax2.bar(labels, mses, color=colors, width=0.5)
    ax2.set_ylabel('Reconstruction MSE')
    ax2.set_title('Reconstruction MSE  (lower = better)')
    ax2.set_ylim(0, max(mses) * 1.25)
    for bar, v in zip(bars2, mses):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(mses) * 0.01,
                 f'{v:.5f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    cmp_path = os.path.join(OUT_DIR, 'comparison.png')
    plt.savefig(cmp_path, dpi=150, bbox_inches='tight')
    print(f'\ncomparison chart saved -> {cmp_path}')
    plt.show()